# GeoMamba — GEOMETRİ TRANSFERİ — rijit cisimde öğren, parçacığa az veriyle uyarla (Kaggle T4)

**Kurulum (bir kere):**
1. Sağdaki panel → **Session options** → Accelerator: **GPU T4**, Internet: **On**
2. Üst menü → **Add-ons → Secrets** → `GH_TOKEN` ekleyin.
   Token: github.com → Settings → Developer settings → Fine-grained tokens →
   sadece `geomamba` reposu, **Contents: Read-only**.
3. **Run All**. Grid ~2-3 saat sürer; sonuçlar `experiments/grid_results.csv`.
4. Bitince **Save Version** → çıktıdan `grid_results.csv`'yi indirin.

**ÖNEMLİ — kalıcılık:** Bu notebook'u interaktif Run All ile DEĞİL,
**Save Version → Save & Run All (Commit)** ile çalıştırın. İnteraktif
session kapanınca çıktılar silinir; Save & Run All çıktıyı kalıcı yapar.

**GPU:** Session options'ta **T4 x2** seçerseniz grid otomatik iki parçaya bölünüp iki GPU'da paralel koşar (süre yarıya iner).


In [1]:
%cd /kaggle/working
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
!rm -rf /kaggle/working/geomamba
!git clone --depth 1 https://{token}@github.com/yusuftiryaki/geomamba.git /kaggle/working/geomamba


/kaggle/working
Cloning into '/kaggle/working/geomamba'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 110 (delta 14), reused 74 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 74.77 KiB | 2.67 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [2]:
%cd /kaggle/working/geomamba
import os, torch
os.environ["PYTHONPATH"] = "/kaggle/working/geomamba/src"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK - Session options'tan GPU secin!")


/kaggle/working/geomamba
GPU: Tesla T4


In [3]:
import os, subprocess, sys, pathlib
GRID = "configs/grid/m4_transfer.yaml"

n_gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ")
print("GPU sayisi:", n_gpu)

# stage 1: pretrain on D1 — must finish before any fine-tune starts
subprocess.run([sys.executable, "scripts/run_grid.py", "--grid", GRID,
                "--only-stage", "1"], check=True)

# stage 2: fine-tunes, one process per GPU
procs = []
for k in range(max(n_gpu, 1)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
    log = open(f"shard{k}.log", "w")
    procs.append((k, log, subprocess.Popen(
        [sys.executable, "scripts/run_grid.py", "--grid", GRID, "--skip-data",
         "--only-stage", "2", "--shard", f"{k}/{max(n_gpu, 1)}"],
        env=env, stdout=log, stderr=subprocess.STDOUT)))
for k, log, p in procs:
    rc = p.wait(); log.close()
    print(f"shard {k} bitti, rc={rc}")
    print(pathlib.Path(f"shard{k}.log").read_text()[-2000:])


GPU sayisi: 2
+ /usr/bin/python3 scripts/generate_data.py --system rigid_body --out data/rigid_body.npz --length 257 --n-train 3000 --n-val 500 --n-test 500
data/rigid_body.npz: train=(3000, 257, 3), train_params=(3000, 3), val=(500, 257, 3), val_params=(500, 3), test=(500, 257, 3), test_params=(500, 3)
+ /usr/bin/python3 scripts/generate_data.py --system lorentz_particle --out data/lorentz_particle.npz --length 257 --n-train 1000 --n-val 500 --n-test 500
data/lorentz_particle.npz: train=(1000, 257, 3), train_params=(1000, 3), val=(500, 257, 3), val_params=(500, 3), test=(500, 257, 3), test_params=(500, 3)
+ /usr/bin/python3 scripts/generate_data.py --system lorentz_particle --out data/lorentz_particle_anom.npz --length 257 --anomalous --n-test 500
data/lorentz_particle_anom.npz: x=(500, 257, 3), anomalous=93/500
+ /usr/bin/python3 scripts/train.py --config configs/exp/xfer_rotor_slot_d1.yaml name=xf-pre-rotorslot seed=0


W0816 20:51:41.441000 84 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


run dir: experiments/runs/xf-pre-rotorslot-20260816-205046  params: 128,928
{'best_val_mse': 4.4647012752344985e-05, 'persistence_mse': 0.000955394372344017, 'steps': 10000, 'wall_seconds': 709.4958901405334}
+ /usr/bin/python3 scripts/evaluate.py --run experiments/runs/xf-pre-rotorslot-20260816-205046
{
  "run": "experiments/runs/xf-pre-rotorslot-20260816-205046",
  "ckpt_step": 10000,
  "test_mse": 6.912243665040781e-05,
  "persistence_mse": 0.0011093496105944118,
  "rollout_mse_h1": 4.99954904474483e-05,
  "rollout_mse_h10": 0.001553036307450384,
  "rollout_mse_h50": 0.05189140260219574,
  "rollout_mse_h100": 0.21182817459106446,
  "energy_drift": 0.9759912490844727,
  "momentum_drift": 0.36576181650161743
}
+ /usr/bin/python3 scripts/train.py --config configs/exp/xfer_mamba_d1.yaml name=xf-pre-mamba seed=0


W0816 21:06:18.773000 581 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


run dir: experiments/runs/xf-pre-mamba-20260816-210603  params: 128,063
{'best_val_mse': 3.8784622120147105e-05, 'persistence_mse': 0.000955394372344017, 'steps': 10000, 'wall_seconds': 123.58751916885376}
+ /usr/bin/python3 scripts/evaluate.py --run experiments/runs/xf-pre-mamba-20260816-210603
{
  "run": "experiments/runs/xf-pre-mamba-20260816-210603",
  "ckpt_step": 10000,
  "test_mse": 5.932666997735699e-05,
  "persistence_mse": 0.0011093496105944118,
  "rollout_mse_h1": 2.517582788641448e-05,
  "rollout_mse_h10": 0.002183514278537283,
  "rollout_mse_h50": 0.1320619308948517,
  "rollout_mse_h100": 0.4102124722798665,
  "energy_drift": 0.5215489864349365,
  "momentum_drift": 0.23223565518856049
}
            name overrides  test_mse  rollout_mse_h100  energy_drift  momentum_drift
xf-pre-rotorslot    seed=0  0.000069          0.211828      0.975991        0.365762
    xf-pre-mamba    seed=0  0.000059          0.410212      0.521549        0.232236
shard 0 bitti, rc=0
pre-mamba-202608

In [4]:
import glob, shutil, pandas as pd
frames = [pd.read_csv(f) for f in sorted(glob.glob("experiments/grid_results*.csv"))]
if frames:
    df = pd.concat(frames, ignore_index=True)
    df.to_csv("/kaggle/working/grid_results.csv", index=False)
    for f in glob.glob("experiments/grid_results*.json"):
        shutil.copy(f, "/kaggle/working/")
    display(df)
else:
    print("sonuc yok - shard*.log dosyalarina bakin")


,run,ckpt_step,test_mse,persistence_mse,rollout_mse_h1,rollout_mse_h10,rollout_mse_h50,rollout_mse_h100,speed_drift,anomaly_auroc,...,event_precision_cusum,event_recall_cusum,anomaly_auroc_inv,event_f1_inv,event_precision_inv,event_recall_inv,name,overrides,energy_drift,momentum_drift
0,experiments/runs/xf-rotorslot-xfer-n25-s0-2026...,3000,0.006160,0.002924,0.004727,0.243106,14.019963,103.785298,51.361320,0.604662,...,0.150000,0.032258,0.513883,0.017857,0.052632,0.010753,xf-rotorslot-xfer-n25-s0,data.n_train=25 seed=0 trainer.steps=3000 trai...,NaN,NaN
1,experiments/runs/xf-mamba-xfer-n25-s0-20260816...,3000,0.000362,0.002924,0.000173,0.030339,1.664238,7.663068,6.077621,0.605705,...,0.300000,0.064516,0.513883,0.017857,0.052632,0.010753,xf-mamba-xfer-n25-s0,data.n_train=25 seed=0 trainer.steps=3000 trai...,NaN,NaN
2,experiments/runs/xf-rotorslot-xfer-n25-s1-2026...,3000,0.005886,0.002924,0.005078,0.239342,13.797968,98.715426,47.551472,0.661121,...,0.062500,0.010753,0.513883,0.017857,0.052632,0.010753,xf-rotorslot-xfer-n25-s1,data.n_train=25 seed=1 trainer.steps=3000 trai...,NaN,NaN
3,experiments/runs/xf-mamba-xfer-n25-s1-20260816...,3000,0.000293,0.002924,0.000206,0.025024,0.941748,1.776773,0.513740,0.637889,...,0.250000,0.043011,0.513883,0.017857,0.052632,0.010753,xf-mamba-xfer-n25-s1,data.n_train=25 seed=1 trainer.steps=3000 trai...,NaN,NaN
4,experiments/runs/xf-rotorslot-xfer-n25-s2-2026...,3000,0.006919,0.002924,0.005755,0.219831,8.983139,67.414233,45.226845,0.635025,...,0.111111,0.021505,0.513883,0.017857,0.052632,0.010753,xf-rotorslot-xfer-n25-s2,data.n_train=25 seed=2 trainer.steps=3000 trai...,NaN,NaN
5,experiments/runs/xf-mamba-xfer-n25-s2-20260816...,3000,0.000325,0.002924,0.000196,0.027920,1.711010,3.841401,1.678754,0.636889,...,0.318182,0.075269,0.513883,0.017857,0.052632,0.010753,xf-mamba-xfer-n25-s2,data.n_train=25 seed=2 trainer.steps=3000 trai...,NaN,NaN
6,experiments/runs/xf-rotorslot-xfer-n100-s0-202...,3000,0.001402,0.002924,0.001009,0.051713,1.956516,13.807847,16.763170,0.704469,...,0.125000,0.021505,0.513883,0.017857,0.052632,0.010753,xf-rotorslot-xfer-n100-s0,data.n_train=100 seed=0 trainer.steps=3000 tra...,NaN,NaN
7,experiments/runs/xf-mamba-xfer-n100-s0-2026081...,3000,0.000147,0.002924,0.000071,0.012152,0.644363,1.408348,0.662597,0.646888,...,0.285714,0.064516,0.513883,0.017857,0.052632,0.010753,xf-mamba-xfer-n100-s0,data.n_train=100 seed=0 trainer.steps=3000 tra...,NaN,NaN
8,experiments/runs/xf-rotorslot-xfer-n100-s1-202...,3000,0.001212,0.002924,0.000957,0.052941,2.468813,16.181623,11.862339,0.701317,...,0.058824,0.010753,0.513883,0.017857,0.052632,0.010753,xf-rotorslot-xfer-n100-s1,data.n_train=100 seed=1 trainer.steps=3000 tra...,NaN,NaN
9,experiments/runs/xf-mamba-xfer-n100-s1-2026081...,3000,0.000118,0.002924,0.000055,0.009948,0.621551,1.269737,0.347174,0.647004,...,0.187500,0.032258,0.513883,0.017857,0.052632,0.010753,xf-mamba-xfer-n100-s1,data.n_train=100 seed=1 trainer.steps=3000 tra...,NaN,NaN
